In [ ]:
import datetime, os
import time
import json
import pandas as pd
import requests
import logging
import threading
import concurrent.futures
import itertools
from urllib.parse import urlparse
from github import Github, Auth
from github.GithubException import BadCredentialsException, UnknownObjectException, GithubException
from pathlib import Path

In [ ]:
from github import Github, Auth
from itertools import cycle


# Comma-separated tokens in your environment:
# export GITHUB_TOKENS="TOKEN1,TOKEN2,TOKEN3"

github_tokens = [t.strip() for t in os.getenv("GITHUB_TOKENS", "").split(",") if t.strip()]

if not github_tokens:
    raise RuntimeError(
        "No GitHub tokens provided. Set GITHUB_TOKENS as a comma-separated list."
    )

In [ ]:
from github import Github, Auth

# Validate and filter valid tokens
valid_tokens = []
for t in github_tokens:
    try:
        g = Github(auth=Auth.Token(t))
        # Minimal, non-identifying check
        print(f"Token OK (core remaining: {rl.remaining})")
        valid_tokens.append(t)
    except BadCredentialsException:
        print("Token rejected")

# Overwrite the original list with valid tokens only
github_tokens = valid_tokens

# Stop execution if no tokens are valid
if not github_tokens:
    raise ValueError("All tokens were rejected.")

In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("github_security_patches.log"),
        logging.StreamHandler()
    ]
)

In [ ]:
class TokenManager:
    """Manages GitHub API tokens with efficient rotation and usage tracking"""
    def __init__(self, tokens):
        self.tokens = tokens
        self.usage = {token: 0 for token in tokens}
        self.lock = threading.Lock()
        
    def get_next_token(self):
        with self.lock:
            # Get token with lowest usage
            token = min(self.usage.items(), key=lambda x: x[1])[0]
            self.usage[token] += 1
            return token
            
    def report_rate_limit(self, token):
        with self.lock:
            # Mark a token as rate-limited by giving it a high usage count
            self.usage[token] = 999999
            logging.warning(f"Token {token[:5]}... marked as rate-limited")

In [ ]:
# Function to extract owner and repo from GitHub URL
def extract_github_owner_repo(repo_url):
    """Extract owner and repository name from a GitHub URL"""
    parsed_url = urlparse(repo_url)
    path_parts = parsed_url.path.strip('/').split('/')
    
    if len(path_parts) < 2:
        return None, None  # Invalid URL
    owner, repo = path_parts[0], path_parts[1]
    return owner, repo.rstrip('.git')  # Remove .git suffix if present

def fetch_commit_patches(repo, commit_sha):
    """Fetch the patch/diff for a specific commit"""
    try:
        commit = repo.get_commit(commit_sha)
        return commit.patch
    except Exception as e:
        logging.error(f"Error fetching patch for commit {commit_sha}: {e}")
        return None

def fetch_code_from_commit(repo, commit_sha, file_path):
    """Fetch file content at specific commit"""
    try:
        contents = repo.get_contents(file_path, ref=commit_sha)
        return contents.decoded_content.decode('utf-8')
    except Exception as e:
        logging.error(f"Error fetching {file_path} at {commit_sha}: {e}")
        return None
    
def find_security_fixes(git_link, repo, start_date=None, max_prs=100):
    """Identify potential security fixes using expanded search criteria"""
    # Use multiple search queries to capture more security-related PRs
    queries = [
        f'repo:{repo.full_name} is:pr is:merged label:security',
        f'repo:{repo.full_name} is:pr is:merged "security fix" in:title',
        f'repo:{repo.full_name} is:pr is:merged "vulnerability" in:title',
        f'repo:{repo.full_name} is:pr is:merged "CVE" in:title',
        f'repo:{repo.full_name} is:pr is:merged "fix" in:title "security" in:body'
    ]

    
    security_fixes = []
    
    for query in queries:
        if start_date:
            query += f' merged:>={start_date.isoformat()}'
        
        try:
            issues = git_link.search_issues(query)
            count = 0
            
            for issue in issues:
                # Skip if we already found this PR (could be returned by multiple queries)
                if any(fix.get('issue') and fix['issue'].number == issue.number for fix in security_fixes):
                    continue
                    
                if count >= max_prs:
                    break
                    
                if issue.pull_request:
                    try:
                        pr = repo.get_pull(issue.number)
                        if pr.merged:
                            security_fixes.append({
                                'issue': issue,
                                'pr': pr,
                                'commit': pr.merge_commit_sha,
                                'files_changed': pr.changed_files
                            })
                            count += 1
                    except GithubException as e:
                        logging.error(f"Error fetching PR #{issue.number}: {e}")
        except GithubException as e:
            logging.error(f"Error searching issues with query '{query}': {e}")
            if "rate limit" in str(e).lower():
                raise e
    
    return security_fixes
    


def analyze_patch_semantics(patch, vulnerable_code, patched_code, commit_message):
    """Analyze patch for security-specific patterns"""
    if not patch:
        return {}
        
    security_keywords = [
        'vulnerability', 'security', 'exploit', 'attack', 'fix', 
        'CVE', 'sanitize', 'validate', 'escape', 'injection', 'overflow',
        'bypass', 'xss', 'csrf', 'ssrf', 'rce', 'dos', 'privilege', 
        'escalation', 'authentication', 'authorization'
    ]
    
    # Check for security keywords
    keyword_matches = [kw for kw in security_keywords if kw.lower() in (patch.lower() or "")]
    message_keywords = [kw for kw in security_keywords if kw.lower() in (commit_message.lower() or "")]
    
    # Simple analysis of changes
    changes = {
        'patch_security_keywords': keyword_matches,
        'message_security_keywords': message_keywords,
        'lines_added': patch.count('\n+') - 1 if patch else 0,  # -1 for the header
        'lines_removed': patch.count('\n-') - 1 if patch else 0,
        'has_input_validation': any(term in patch.lower() for term in ['validate', 'sanitize', 'escape']),
        'has_cve_reference': 'cve-' in (commit_message.lower() or ""),
        'confidence_score': len(keyword_matches) + len(message_keywords) * 2  # Simple scoring
    }
    
    return changes

    
def process_repository(git_link, repo_url, save_intermediate=False):
    """Process a single repository to find vulnerable/patched pairs"""
    
    logging.info(f"Starting processing: {repo_url}")
    owner, project = extract_github_owner_repo(repo_url)
    if not owner or not project:
        logging.warning(f"Invalid GitHub URL format: {repo_url}")
        return None
    
    try:
        logging.info(f"Fetching repo: {owner}/{project}")
        repo = git_link.get_repo(f"{owner}/{project}")
        logging.info(f"Repo found: {repo.full_name}")
        
        # Get metadata
        metadata = {
            'repo_url': repo_url,
            'repo_name': repo.full_name,
            'description': repo.description,
            'date_created': repo.created_at,
            'date_last_push': repo.pushed_at,
            'homepage': repo.homepage,
            'repo_language': repo.language,
            'forks_count': repo.forks,
            'stars_count': repo.stargazers_count,
            'issues_count': repo.open_issues_count,
            'owner': owner,
            'analysis_date': datetime.datetime.now()
        }
        
        # Find security-related fixes
        logging.info(f"Looking for security fixes in {repo.full_name}...")
        security_fixes = find_security_fixes(git_link, repo, start_date=repo.created_at)
        logging.info(f"Found {len(security_fixes)} possible security Pull Requests.")
        
        patches = []
        for fix_idx, fix in enumerate(security_fixes):
            try:
                commit = repo.get_commit(fix['commit'])
                
                # Get parent commit (vulnerable version)
                parent_commit = commit.parents[0] if commit.parents else None
                if not parent_commit:
                    continue
                
                # Process each changed file
                for file in commit.files:
                    # Skip non-code files
                    if not (file.filename.endswith('.py') or file.filename.endswith('.java') or 
                          file.filename.endswith('.c') or file.filename.endswith('.cpp') or
                          file.filename.endswith('.js') or file.filename.endswith('.php')): 
                        continue
                    
                    # Get vulnerable and patched versions
                    vulnerable_code = fetch_code_from_commit(repo, parent_commit.sha, file.filename)
                    patched_code = fetch_code_from_commit(repo, commit.sha, file.filename)
                    
                    if vulnerable_code and patched_code:
                        # Analyze the semantics of the patch
                        semantics = analyze_patch_semantics(
                            file.patch, 
                            vulnerable_code, 
                            patched_code, 
                            commit.commit.message
                        )
                        
                        patch_record = {
                            'repo_url': repo_url,
                            'commit_sha': commit.sha,
                            'parent_sha': parent_commit.sha,
                            'filename': file.filename,
                            'patch': file.patch,
                            'vulnerable_code': vulnerable_code,
                            'patched_code': patched_code,
                            'commit_message': commit.commit.message,
                            'commit_date': commit.commit.author.date,
                            'fix_issue': fix['issue'].title if fix['issue'] else None,
                            'file_extension': Path(file.filename).suffix,
                            'lines_added': semantics.get('lines_added', 0),
                            'lines_removed': semantics.get('lines_removed', 0),
                            'has_input_validation': semantics.get('has_input_validation', False),
                            'has_cve_reference': semantics.get('has_cve_reference', False),
                            'security_keywords': ','.join(semantics.get('patch_security_keywords', [])),
                            'message_keywords': ','.join(semantics.get('message_security_keywords', [])),
                            'confidence_score': semantics.get('confidence_score', 0)
                        }
                        patches.append(patch_record)
                
                # Save intermediate results every 5 fixes if enabled
                #if save_intermediate and fix_idx > 0 and fix_idx % 5 == 0:
                    #save_intermediate_results([metadata], patches, prefix=f"{owner}_{project}_")
                    
            except GithubException as e:
                logging.error(f"Error processing commit {fix['commit']}: {e}")
                if "rate limit" in str(e).lower():
                    raise e  # Re-raise to handle in outer function
        
        return {
            'status': 'success',
            'metadata': metadata,
            'patches': patches
        }
        
    except UnknownObjectException:
        logging.error(f"Repository {repo_url} not found (404). It may be private or deleted.")
        return {'status': 'not_found'}
    except GithubException as e:
        logging.error(f"GitHub API error for {repo_url}: {e}")
        if "rate limit" in str(e).lower():
            return {'status': 'rate_limited'}
    except Exception as e:
        logging.error(f"Unexpected error for {repo_url}: {e}", exc_info=True)
        return {'status': 'error', 'error': str(e)}
    
    return {'status': 'error', 'error': 'Unknown error'}

def fetch_github_patches(available_repo_urls, token_list, max_workers=5, save_intermediate=True):
    """Process multiple repositories in parallel with token rotation"""
    token_manager = TokenManager(token_list)
    metadata_list = []
    patches_list = []
    results_lock = threading.Lock()
    
    def _threaded_process_repository(repo_url):
        token = token_manager.get_next_token()
        logging.info(f"Using token: {token[:5]}... for {repo_url}")
        
        try:
            git_link = Github(auth=Auth.Token(token))
            result = process_repository(git_link, repo_url, save_intermediate=False)
            
            if result:
                if result['status'] == 'success':
                    with results_lock:
                        metadata_list.append(result['metadata'])
                        patches_list.extend(result['patches'])
                elif result['status'] == 'rate_limited':
                    token_manager.report_rate_limit(token)
                    # Retry with a different token
                    logging.info(f"Retrying {repo_url} with a different token due to rate limit")
                    return _threaded_process_repository(repo_url)
            
            return result
        except Exception as e:
            logging.error(f"Thread error processing {repo_url}: {e}", exc_info=True)
            return {'status': 'error', 'error': str(e)}
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(_threaded_process_repository, url): url for url in available_repo_urls}
        
        completed = 0
        total = len(available_repo_urls)
        for future in concurrent.futures.as_completed(future_to_url):
            url = future_to_url[future]
            try:
                result = future.result()
                completed += 1
                logging.info(f"Completed {completed}/{total}: {url} - Status: {result['status'] if result else 'None'}")
                
                # Save intermediate results every 5 repositories
                #if save_intermediate and completed % 5 == 0:
                    #save_intermediate_results(metadata_list, patches_list)
                    
            except Exception as e:
                logging.error(f"Error processing {url}: {e}", exc_info=True)
    
    # Create final DataFrames
    repo_columns = [
        'repo_url', 'repo_name', 'description', 'date_created', 'date_last_push',
        'homepage', 'repo_language', 'forks_count', 'stars_count', 'issues_count', 
        'owner', 'analysis_date'
    ]
    
    patch_columns = [
        'repo_url', 'commit_sha', 'parent_sha', 'filename', 'file_extension',
        'patch', 'vulnerable_code', 'patched_code', 'commit_message', 
        'commit_date', 'fix_issue', 'lines_added', 'lines_removed',
        'has_input_validation', 'has_cve_reference', 'security_keywords',
        'message_keywords', 'confidence_score'
    ]
    
    df_metadata = pd.DataFrame(metadata_list, columns=repo_columns)
    df_patches = pd.DataFrame(patches_list, columns=patch_columns)
    
    logging.info(f'Collected metadata for {len(df_metadata)} repositories and {len(df_patches)} patches.')
    return df_metadata, df_patches


In [ ]:
csv_path = Path("NVD_Data")
csv_file_path = csv_path / "available_urls.csv"
print(csv_file_path)

In [ ]:
# Load the CSV file
available_urls_df = pd.read_csv(csv_file_path)


total_urls = len(available_urls_df)
print(total_urls)

start = -50000
end = -48000  # for last 2000 URLs
available_urls = available_urls_df.iloc[:, 0].dropna().tolist()[start:end]


print(f"Processing {len(available_urls)} repositories...")
#for url in available_urls:
    #print(" -", url)

df_metadata, df_patches = fetch_github_patches(available_urls, github_tokens, max_workers=5)


batch_label = f"urls_{start}_to_{end if end is not None else 'end'}"
timestamp = datetime.datetime.now().strftime("%d.%m.%Y")
base_output_dir = Path("final_output") / f"reports_{batch_label}_{timestamp}"
base_output_dir.mkdir(parents=True, exist_ok=True)
    
    
metadata_path = base_output_dir / f"repo_metadata_{timestamp}.csv"
patches_path = base_output_dir / f"security_patches_{timestamp}.csv"

df_metadata.to_csv(metadata_path, index=False)
df_patches.to_csv(patches_path, index=False)

df_metadata.to_pickle(base_output_dir / f"repo_metadata_{timestamp}.pkl")
df_patches.to_pickle(base_output_dir / f"security_patches_{timestamp}.pkl")

logging.info(f"Saved results to {metadata_path} and {patches_path}")

    

In [ ]:
# Generate a summary report

with open(base_output_dir / f"summary_report_{timestamp}.txt", 'w') as f:
    f.write("GitHub Security Patches Collection Report\n")
    f.write(f"Generated: {datetime.datetime.now()}\n\n")
    f.write(f"Repositories processed: {len(df_metadata)}\n")
    f.write(f"Security patches collected: {len(df_patches)}\n\n")

    if not df_patches.empty:
        f.write("Top repositories by patch count:\n")
        repo_counts = df_patches['repo_url'].value_counts().head(10)
        for repo, count in repo_counts.items():
            f.write(f"  - {repo}: {count} patches\n")

        f.write("\nFile extensions distribution:\n")
        ext_counts = df_patches['file_extension'].value_counts()
        for ext, count in ext_counts.items():
            f.write(f"  - {ext}: {count} files\n")

        f.write(f"\nHigh confidence patches (score >= 5): {len(df_patches[df_patches['confidence_score'] >= 5])}\n")

logging.info(f"Summary report generated at {base_output_dir / f'summary_report_{timestamp}.txt'}")


Key Improvements

- Better Token Management
    Created a TokenManager class that tracks usage and prioritizes less-used tokens
    Handles rate limiting more efficiently by marking rate-limited tokens


- Robust Error Handling and Logging
    Added comprehensive logging with both file and console output
    Better exception handling throughout the code
    Detailed error messages with stack traces for debugging


- Intermediate Results Saving
    Automatically saves data periodically to prevent loss from crashes
    Checkpoints after processing batches of repositories


- Enhanced Security Analysis
    Added analyze_patch_semantics() function to detect security patterns
    Calculates a confidence score for each patch based on keywords
    Tracks additional metrics like presence of CVE references


- Data Processing Improvements
    Better handling of pagination for repositories with many PRs
    Added file extension filtering and statistics
    Saves results in both CSV and pickle formats for better data preservation


- Progress Tracking
    Shows progress during processing (completed/total)
    Generates a summary report with key statistics


- Configuration Options
    Allows loading repository URLs from a file
    Configurable number of worker threads
    Option to enable/disable intermediate saving

In [ ]:
def save_vulnerable_code_snippets(patches_df: pd.DataFrame, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    """
    Save vulnerable_code rows to dated folder, and write a manifest.csv mapping
    safe_filename → repo_url, commit_sha, original filename, and saved path.
    """

    stats = {"total": len(patches_df), "saved": 0, "skipped": 0}
    manifest = []

    for idx, row in patches_df.iterrows():
        vulnerable = row.get('vulnerable_code')
        if not vulnerable or pd.isna(vulnerable):
            stats["skipped"] += 1
            continue

        ext = row.get('file_extension') or Path(row.get('filename', '')).suffix
        if ext and not ext.startswith('.'):
            ext = '.' + ext

        repo_name = Path(row['repo_url']).stem
        sha7 = (row.get('commit_sha') or f"idx{idx}")[:7]
        original = Path(row.get('filename', f"file_{idx}")).name

        base = f"{repo_name}_{sha7}_{original}"
        safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in base)
        if ext and not safe.endswith(ext):
            safe += ext

        out_path = output_dir / safe

        try:
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(vulnerable)
            stats["saved"] += 1

            manifest.append({
                "safe_filename": safe,
                "file_path": str(out_path),
                "repo_url": row['repo_url'],
                "commit_sha": row.get('commit_sha'),
                "original_filename": row.get('filename'),
            })
        except Exception as e:
            logging.error(f"Error at row {idx}: {e}", exc_info=True)
            stats["skipped"] += 1

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(base_output_dir / "snippets_filename_metadata_vulncode.csv", index=False)

    logging.info(f"Vulnerable snippets: saved {stats['saved']} / {stats['total']} → {output_dir}")
    return stats, manifest_df


In [ ]:
def save_patches(patches_df: pd.DataFrame, output_dir: Path, extension=".patch"):
    output_dir.mkdir(parents=True, exist_ok=True)
    """
    Save patch text to dated folder + manifest.csv mapping safe_filename → source info.
    """

    stats = {"total": len(patches_df), "saved": 0, "skipped": 0}
    manifest = []

    for idx, row in patches_df.iterrows():
        patch = row.get('patch')
        if not patch or pd.isna(patch):
            stats["skipped"] += 1
            continue

        repo_name = Path(row['repo_url']).stem
        sha7 = (row.get('commit_sha') or f"idx{idx}")[:7]
        base = f"{repo_name}_{sha7}"
        safe = "".join(c if c.isalnum() or c in "_-" else "_" for c in base)
        filename = safe + extension

        out_path = output_dir / filename  

        try:
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(patch)
            stats["saved"] += 1

            manifest.append({
                "safe_filename": filename,
                "file_path": str(out_path),
                "repo_url": row['repo_url'],
                "commit_sha": row.get('commit_sha'),
            })
        except Exception as e:
            logging.error(f"Error saving patch at row {idx}: {e}", exc_info=True)
            stats["skipped"] += 1

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(base_output_dir / "snippets_metadata_patches.csv", index=False)  

    logging.info(f"Patches: saved {stats['saved']} / {stats['total']} → {output_dir}")
    return stats, manifest_df


In [ ]:
# Save just the vulnerable code to separate files


vuln_dir = base_output_dir / "vulnerable_code"
patch_dir = base_output_dir / "patches"

patch_stats, patch_manifest = save_patches(df_patches, patch_dir)
vuln_stats, vuln_manifest = save_vulnerable_code_snippets(df_patches, vuln_dir)


In [ ]:
import re
from datetime import datetime
from collections import defaultdict

# Function to search commits for CVE references
def search_cve_commits(repo_url, cve_id, token):
    """Search commits in a repository that reference a specific CVE"""
    owner, repo_name = extract_github_owner_repo(repo_url)
    if not owner or not repo_name:
        return []
    
    g = Github(token)
    repo = g.get_repo(f"{owner}/{repo_name}")
    
    # Search commit messages for CVE ID
    query = f"{cve_id} repo:{owner}/{repo_name}"
    commits = []
    
    try:
        commits = list(g.search_commits(query=query))
    except GithubException as e:
        print(f"Error searching commits for {cve_id} in {repo_url}: {e}")
    
    return commits

# Function to extract patch information from a commit
def extract_patch_info(commit, cve_id):
    """Extract relevant patch information from a commit"""
    patch_info = {
        'cve_id': cve_id,
        'commit_hash': commit.sha,
        'commit_message': commit.commit.message,
        'commit_date': commit.commit.committer.date,
        'author': commit.author.login if commit.author else None,
        'files_changed': commit.files,
        'patch_details': []
    }
    
    for file in commit.files:
        try:
            patch_info['patch_details'].append({
                'filename': file.filename,
                'status': file.status,
                'additions': file.additions,
                'deletions': file.deletions,
                'patch': file.patch,
                'language': file.filename.split('.')[-1] if '.' in file.filename else None
            })
        except Exception as e:
            print(f"Error processing file {file.filename} in commit {commit.sha}: {e}")
    
    return patch_info

# Function to identify vulnerable and patched code
def extract_code_snippets(patch_info):
    """Extract vulnerable and patched code snippets from patch details"""
    snippets = []
    
    for file in patch_info['patch_details']:
        if not file.get('patch'):
            continue
            
        # Parse unified diff to get before/after code
        changes = parse_diff(file['patch'])
        if changes:
            for change in changes:
                snippets.append({
                    'filename': file['filename'],
                    'language': file['language'],
                    'vulnerable_code': change.get('before'),
                    'patched_code': change.get('after'),
                    'change_type': change.get('change_type'),
                    'context': change.get('context')
                })
    
    return snippets

# Helper function to parse unified diff format
def parse_diff(patch_text):
    """Parse unified diff to extract before/after code snippets"""
    if not patch_text:
        return []
    
    changes = []
    current_change = {'before': [], 'after': [], 'context': []}
    
    for line in patch_text.split('\n'):
        if line.startswith('@@'):
            # New hunk - save previous change if any
            if current_change['before'] or current_change['after']:
                changes.append(current_change)
            current_change = {'before': [], 'after': [], 'context': []}
        elif line.startswith('-'):
            current_change['before'].append(line[1:])
        elif line.startswith('+'):
            current_change['after'].append(line[1:])
        else:
            current_change['context'].append(line[1:])
    
    if current_change['before'] or current_change['after']:
        changes.append(current_change)
    
    return changes

# Main function to process all CVEs and repositories
def process_vulnerability_patches(df_cve, df_repos, token, output_path):
    """Main function to extract vulnerability patches from GitHub"""
    results = []
    
    # Create a mapping from repo URL to CVEs
    repo_cve_map = defaultdict(list)
    for _, row in df_cve.iterrows():
        if not isinstance(row['reference_json'], list):
            continue
        for url in extract_github_urls(row['reference_json']):
            repo_cve_map[url].append(row['cve_id'])
    
    # Process each repository
    for repo_url, cve_ids in repo_cve_map.items():
        print(f"\nProcessing repository: {repo_url}")
        
        # Check if this repo is in our available repos
        repo_meta = df_repos[df_repos['repo_url'] == repo_url]
        if repo_meta.empty:
            print(f"Skipping {repo_url} - not in available repositories")
            continue
            
        for cve_id in cve_ids:
            print(f"Searching for patches for {cve_id}")
            
            commits = search_cve_commits(repo_url, cve_id, token)
            if not commits:
                print(f"No commits found for {cve_id} in {repo_url}")
                continue
                
            for commit in commits:
                try:
                    patch_info = extract_patch_info(commit, cve_id)
                    code_snippets = extract_code_snippets(patch_info)
                    
                    if code_snippets:
                        result = {
                            'cve_id': cve_id,
                            'repo_url': repo_url,
                            'commit_hash': patch_info['commit_hash'],
                            'commit_date': patch_info['commit_date'],
                            'commit_message': patch_info['commit_message'],
                            'code_snippets': code_snippets,
                            'repo_metadata': repo_meta.iloc[0].to_dict()
                        }
                        results.append(result)
                        
                        # Save progress after each CVE
                        df_results = pd.DataFrame(results)
                        df_results.to_json(f"{output_path}/vulnerability_patches.json", orient='records', lines=True)
                        
                except Exception as e:
                    print(f"Error processing commit {commit.sha} for {cve_id}: {e}")
    
    print(f"\nCompleted processing. Saved results to {output_path}/vulnerability_patches.json")
    return results


